# 1. Import Libraries

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# 2. Define Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

MONTHLY_KPI_PATH = PROCESSED_DATA_DIR / "monthly_kpi.csv"
BUDGET_ACTUAL_PATH = PROCESSED_DATA_DIR / "budget_actual.csv"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", PROCESSED_DATA_DIR)

Project root: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new
Processed data dir: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new\data\processed


# 3. Load Monthly KPI and Budget Data

In [3]:
monthly_kpi = pd.read_csv(MONTHLY_KPI_PATH)
budget_actual = pd.read_csv(BUDGET_ACTUAL_PATH)

print("Monthly KPI:", monthly_kpi.shape)
print("Budget Actual:", budget_actual.shape)

monthly_kpi.head()

Monthly KPI: (36, 23)
Budget Actual: (36, 28)


,Year,Quarter,Month,Month_Name,Year_Month,order_count,units_sold,gross_sales,net_revenue,cogs,gross_profit,logistics_cost,marketing_spend,profit,contribution_profit,avg_discount_pct,profit_margin,gross_margin,contribution_margin,cogs_ratio,logistics_ratio,marketing_ratio,avg_selling_price
0,2023,Q1,1,January,2023-01,535,110629,"481,511.88","408,301.70","242,713.37","165,588.33","27,940.41","43,286.62","94,361.30","94,361.30",12.36,0.23,0.41,0.23,0.59,0.07,0.11,3.69
1,2023,Q1,2,February,2023-02,472,98261,"419,997.02","354,938.26","210,194.58","144,743.68","24,395.52","39,437.88","80,910.28","80,910.28",13.14,0.23,0.41,0.23,0.59,0.07,0.11,3.61
2,2023,Q1,3,March,2023-03,512,105895,"438,332.68","371,116.33","222,070.30","149,046.03","24,777.09","40,046.44","84,222.50","84,222.50",12.98,0.23,0.40,0.23,0.60,0.07,0.11,3.50
3,2023,Q2,4,April,2023-04,497,100548,"421,101.02","357,347.48","212,010.72","145,336.76","25,161.47","38,509.72","81,665.57","81,665.57",12.67,0.23,0.41,0.23,0.59,0.07,0.11,3.55
4,2023,Q2,5,May,2023-05,479,99014,"427,223.81","360,818.12","216,709.26","144,108.86","24,584.69","38,239.59","81,284.58","81,284.58",12.96,0.23,0.40,0.23,0.60,0.07,0.11,3.64


# 4. Monthly Time Series

In [4]:
ts = monthly_kpi.copy()

ts["date"] = pd.to_datetime(ts["Year_Month"] + "-01")
ts = ts.sort_values("date").reset_index(drop=True)

ts = ts[[
    "date",
    "Year",
    "Quarter",
    "Month",
    "Month_Name",
    "Year_Month",
    "net_revenue",
    "profit",
    "units_sold",
    "marketing_spend",
    "profit_margin",
    "marketing_ratio",
]]

ts = ts.rename(columns={"net_revenue": "actual_net_revenue"})

ts.head()

,date,Year,Quarter,Month,Month_Name,Year_Month,actual_net_revenue,profit,units_sold,marketing_spend,profit_margin,marketing_ratio
0,2023-01-01,2023,Q1,1,January,2023-01,"408,301.70","94,361.30",110629,"43,286.62",0.23,0.11
1,2023-02-01,2023,Q1,2,February,2023-02,"354,938.26","80,910.28",98261,"39,437.88",0.23,0.11
2,2023-03-01,2023,Q1,3,March,2023-03,"371,116.33","84,222.50",105895,"40,046.44",0.23,0.11
3,2023-04-01,2023,Q2,4,April,2023-04,"357,347.48","81,665.57",100548,"38,509.72",0.23,0.11
4,2023-05-01,2023,Q2,5,May,2023-05,"360,818.12","81,284.58",99014,"38,239.59",0.23,0.11


In [5]:
# Validate time-series continuity

expected_dates = pd.date_range(
    start=ts["date"].min(),
    end=ts["date"].max(),
    freq="MS"
)

missing_dates = sorted(set(expected_dates) - set(ts["date"]))

print("Start date:", ts["date"].min().date())
print("End date:", ts["date"].max().date())
print("Total months:", len(ts))
print("Missing months:", len(missing_dates))

missing_dates[:10]

Start date: 2023-01-01
End date: 2025-12-01
Total months: 36
Missing months: 0


[]

# 5. Train/Test Split

In [6]:
TEST_MONTH = 6

train = ts.iloc[:-TEST_MONTH].copy()
test = ts.iloc[-TEST_MONTH:].copy()

print("Train period:", train["date"].min().date(), "to", train["date"].max().date())
print("Test period:", test["date"].min().date(), "to", test["date"].max().date())
print("Train months:", len(train))
print("Test months:", len(test))

Train period: 2023-01-01 to 2025-06-01
Test period: 2025-07-01 to 2025-12-01
Train months: 30
Test months: 6


In [7]:
# Define evaluation functions

def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))


def evaluate_forecast(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
    }

# 7. Model Development 1: Seasonal Naive (Baseline Model)

Forecast = actual value from same month last year

In [8]:
seasonal_lookup = train[["Month", "actual_net_revenue"]].copy()

# Use the latest available same-month value from the training period
seasonal_lookup = (
    seasonal_lookup
    .groupby("Month", as_index=False)
    .tail(1)
    .set_index("Month")["actual_net_revenue"]
    .to_dict()
)

fallback_value = train["actual_net_revenue"].mean()

test["seasonal_naive_forecast"] = test["Month"].map(seasonal_lookup)
test["seasonal_naive_forecast"] = test["seasonal_naive_forecast"].fillna(fallback_value)

test[["date", "Month_Name", "actual_net_revenue", "seasonal_naive_forecast"]]

,date,Month_Name,actual_net_revenue,seasonal_naive_forecast
30,2025-07-01,July,"373,404.97","399,964.40"
31,2025-08-01,August,"454,648.33","372,987.42"
32,2025-09-01,September,"413,041.17","411,851.11"
33,2025-10-01,October,"388,158.81","399,160.10"
34,2025-11-01,November,"352,684.08","404,944.21"
35,2025-12-01,December,"490,724.29","456,863.41"


# 8. Model Development 2: Movign Average

In [9]:
def recursive_moving_average_forecast(train_series, horizon, window=3):
    history = list(train_series)
    forecasts = []
    
    for _ in range(horizon):
        forecast = np.mean(history[-window:])
        forecasts.append(forecast)
        history.append(forecast)
    
    return np.array(forecasts)


test["moving_average_forecast"] = recursive_moving_average_forecast(
    train["actual_net_revenue"].values,
    horizon=len(test),
    window=3
)

test[["date", "actual_net_revenue", "moving_average_forecast"]]

,date,actual_net_revenue,moving_average_forecast
30,2025-07-01,"373,404.97","424,551.23"
31,2025-08-01,"454,648.33","422,792.51"
32,2025-09-01,"413,041.17","412,127.46"
33,2025-10-01,"388,158.81","419,823.73"
34,2025-11-01,"352,684.08","418,247.90"
35,2025-12-01,"490,724.29","416,733.03"


# 9. Model Development 2: Holt-Winters Exponential Smoothing

In [10]:
hw_train = train.set_index("date")["actual_net_revenue"]
hw_test_index = test["date"]

holt_winters_model = ExponentialSmoothing(
    hw_train,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
)

holt_winters_fit = holt_winters_model.fit(optimized=True)

test["holt_winters_forecast"] = holt_winters_fit.forecast(len(test)).values

test[["date", "actual_net_revenue", "holt_winters_forecast"]]

,date,actual_net_revenue,holt_winters_forecast
30,2025-07-01,"373,404.97","419,409.19"
31,2025-08-01,"454,648.33","421,596.21"
32,2025-09-01,"413,041.17","414,986.00"
33,2025-10-01,"388,158.81","434,779.05"
34,2025-11-01,"352,684.08","419,932.31"
35,2025-12-01,"490,724.29","467,088.24"


# 10. Model Development 4: SARIMA

In [11]:
sarima_train = train.set_index("date")["actual_net_revenue"]

sarima_candidates = [
    {
        "name": "SARIMA(1,1,1)(1,1,1,12)",
        "order": (1, 1, 1),
        "seasonal_order": (1, 1, 1, 12),
    },
    {
        "name": "SARIMA(0,1,1)(0,1,1,12)",
        "order": (0, 1, 1),
        "seasonal_order": (0, 1, 1, 12),
    },
    {
        "name": "SARIMA(1,1,0)(1,1,0,12)",
        "order": (1, 1, 0),
        "seasonal_order": (1, 1, 0, 12),
    },
    {
        "name": "ARIMA(1,1,1)",
        "order": (1, 1, 1),
        "seasonal_order": (0, 0, 0, 0),
    },
]

In [12]:
# Fit SARIMA candidates and select best

sarima_results = []
sarima_forecasts = {}

for candidate in sarima_candidates:
    model_name = candidate["name"]
    order = candidate["order"]
    seasonal_order = candidate["seasonal_order"]
    
    try:
        model = SARIMAX(
            sarima_train,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        
        fit = model.fit(disp=False)
        forecast = fit.forecast(steps=len(test))
        
        metrics = evaluate_forecast(
            test["actual_net_revenue"],
            forecast.values,
            model_name,
        )
        
        metrics["AIC"] = fit.aic
        metrics["BIC"] = fit.bic
        sarima_results.append(metrics)
        sarima_forecasts[model_name] = forecast.values
        
        print(f"Success: {model_name}")
        
    except Exception as e:
        print(f"Failed: {model_name} | Error: {e}")

sarima_metrics = pd.DataFrame(sarima_results).sort_values("MAPE").reset_index(drop=True)
sarima_metrics

Success: SARIMA(1,1,1)(1,1,1,12)
Success: SARIMA(0,1,1)(0,1,1,12)
Success: SARIMA(1,1,0)(1,1,0,12)
Success: ARIMA(1,1,1)


,model,MAE,RMSE,MAPE,AIC,BIC
0,"SARIMA(0,1,1)(0,1,1,12)","38,148.42","47,558.61",0.09,73.98,71.28
1,"SARIMA(1,1,0)(1,1,0,12)","39,238.67","47,428.55",0.10,98.88,97.04
2,"SARIMA(1,1,1)(1,1,1,12)","43,283.98","51,728.34",0.11,77.73,73.22
3,"ARIMA(1,1,1)","44,655.53","49,669.03",0.11,629.97,633.86


In [13]:
# Select SARIMA candidate based on out-of-sample MAPE.
# AIC/BIC are retained for reference, but the final selection prioritizes
# test-period forecasting performance because the time series is short.

sarima_metrics = sarima_metrics.sort_values(["MAPE", "RMSE", "MAE"]).reset_index(drop=True)

sarima_metrics

,model,MAE,RMSE,MAPE,AIC,BIC
0,"SARIMA(0,1,1)(0,1,1,12)","38,148.42","47,558.61",0.09,73.98,71.28
1,"SARIMA(1,1,0)(1,1,0,12)","39,238.67","47,428.55",0.10,98.88,97.04
2,"SARIMA(1,1,1)(1,1,1,12)","43,283.98","51,728.34",0.11,77.73,73.22
3,"ARIMA(1,1,1)","44,655.53","49,669.03",0.11,629.97,633.86


### SARIMA Candidate Selection

All SARIMA candidates were successfully fitted.  
The best SARIMA specification is selected based on out-of-sample MAPE on the last 6 months of data.

Although AIC and BIC are reported for reference, the final model comparison prioritizes test-period forecast accuracy because the available monthly history is relatively short.

In [14]:
# Add best SARIMA forecast to test set

if not sarima_metrics.empty:
    best_sarima_model = sarima_metrics.iloc[0]["model"]
    test["sarima_forecast"] = sarima_forecasts[best_sarima_model]
    print("Best SARIMA model:", best_sarima_model)
else:
    best_sarima_model = None
    test["sarima_forecast"] = np.nan
    print("No SARIMA model succeeded.") 

Best SARIMA model: SARIMA(0,1,1)(0,1,1,12)


# 11. Model Evaluation

In [15]:
# Evaluate all models

model_metrics = []

model_metrics.append(
    evaluate_forecast(
        test["actual_net_revenue"],
        test["seasonal_naive_forecast"],
        "Seasonal Naive",
    )
)

model_metrics.append(
    evaluate_forecast(
        test["actual_net_revenue"],
        test["moving_average_forecast"],
        "Moving Average (3M)",
    )
)

model_metrics.append(
    evaluate_forecast(
        test["actual_net_revenue"],
        test["holt_winters_forecast"],
        "Holt-Winters",
    )
)

if test["sarima_forecast"].notna().all():
    model_metrics.append(
        evaluate_forecast(
            test["actual_net_revenue"],
            test["sarima_forecast"],
            best_sarima_model,
        )
    )

forecast_metrics = pd.DataFrame(model_metrics)
forecast_metrics = forecast_metrics.sort_values("MAPE").reset_index(drop=True)

forecast_metrics

,model,MAE,RMSE,MAPE
0,Seasonal Naive,"34,422.12","43,539.32",0.08
1,"SARIMA(0,1,1)(0,1,1,12)","38,148.42","47,558.61",0.09
2,Holt-Winters,"36,417.62","41,767.31",0.09
3,Moving Average (3M),"42,522.63","49,002.65",0.10


In [16]:
# Select best model
best_model_name = forecast_metrics.iloc[0]["model"]
best_model_mape = forecast_metrics.iloc[0]["MAPE"]

print("Best model:", best_model_name)
print(f"Best test MAPE: {best_model_mape:.2%}")

Best model: Seasonal Naive
Best test MAPE: 8.32%


### Forecast Model Selection Result

The Seasonal Naive model achieved the lowest test MAPE among the evaluated forecasting methods.

This result suggests that monthly net revenue is strongly influenced by recurring seasonal patterns. Since the available monthly history is relatively short, a simple seasonal benchmark can outperform more parameterized models such as Holt-Winters and SARIMA.

For business planning, the final model is selected based on out-of-sample test performance rather than model complexity.

In [17]:
# Forecast results table

forecast_results = test[[
    "date",
    "Year",
    "Quarter",
    "Month",
    "Month_Name",
    "Year_Month",
    "actual_net_revenue",
    "seasonal_naive_forecast",
    "moving_average_forecast",
    "holt_winters_forecast",
    "sarima_forecast",
]].copy()

forecast_results["best_model"] = best_model_name

if best_model_name == "Seasonal Naive":
    forecast_results["best_model_forecast"] = forecast_results["seasonal_naive_forecast"]
elif best_model_name == "Moving Average (3M)":
    forecast_results["best_model_forecast"] = forecast_results["moving_average_forecast"]
elif best_model_name == "Holt-Winters":
    forecast_results["best_model_forecast"] = forecast_results["holt_winters_forecast"]
else:
    forecast_results["best_model_forecast"] = forecast_results["sarima_forecast"]

forecast_results["forecast_error"] = (
    forecast_results["actual_net_revenue"] - forecast_results["best_model_forecast"]
)
forecast_results["forecast_error_pct"] = (
    forecast_results["forecast_error"] / forecast_results["actual_net_revenue"].replace(0, np.nan)
)

forecast_results

,date,Year,Quarter,Month,Month_Name,Year_Month,actual_net_revenue,seasonal_naive_forecast,moving_average_forecast,holt_winters_forecast,sarima_forecast,best_model,best_model_forecast,forecast_error,forecast_error_pct
30,2025-07-01,2025,Q3,7,July,2025-07,"373,404.97","399,964.40","424,551.23","419,409.19","405,435.60",Seasonal Naive,"399,964.40","-26,559.43",-0.07
31,2025-08-01,2025,Q3,8,August,2025-08,"454,648.33","372,987.42","422,792.51","421,596.21","364,973.67",Seasonal Naive,"372,987.42","81,660.91",0.18
32,2025-09-01,2025,Q3,9,September,2025-09,"413,041.17","411,851.11","412,127.46","414,986.00","424,524.13",Seasonal Naive,"411,851.11","1,190.06",0.00
33,2025-10-01,2025,Q4,10,October,2025-10,"388,158.81","399,160.10","419,823.73","434,779.05","394,877.90",Seasonal Naive,"399,160.10","-11,001.29",-0.03
34,2025-11-01,2025,Q4,11,November,2025-11,"352,684.08","404,944.21","418,247.90","419,932.31","410,729.53",Seasonal Naive,"404,944.21","-52,260.13",-0.15
35,2025-12-01,2025,Q4,12,December,2025-12,"490,724.29","456,863.41","416,733.03","467,088.24","459,786.59",Seasonal Naive,"456,863.41","33,860.88",0.07


In [19]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=forecast_results["date"],
        y=forecast_results["actual_net_revenue"],
        mode="lines+markers",
        name="Actual Net Revenue",
    )
)

fig.add_trace(
    go.Scatter(
        x=forecast_results["date"],
        y=forecast_results["best_model_forecast"],
        mode="lines+markers",
        name=f"Forecast ({best_model_name})",
    )
)

fig.update_layout(
    title=f"Actual vs Forecast Net Revenue — Test Period ({best_model_name})",
    xaxis_title="Month",
    yaxis_title="Net Revenue USD",
)

fig.show()

## ML Model Quick Testing

In [20]:
# Prepare leakage-safe ML features
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ml_df = ts[["date", "Year", "Quarter", "Month", "Month_Name", "Year_Month", "actual_net_revenue"]].copy()
ml_df = ml_df.sort_values("date").reset_index(drop=True)

# Time index
ml_df["time_idx"] = np.arange(len(ml_df))

# Calendar features
ml_df["quarter_num"] = pd.to_datetime(ml_df["date"]).dt.quarter
ml_df["month_sin"] = np.sin(2 * np.pi * ml_df["Month"] / 12)
ml_df["month_cos"] = np.cos(2 * np.pi * ml_df["Month"] / 12)

# Leakage-safe lag features
ml_df["lag_1"] = ml_df["actual_net_revenue"].shift(1)
ml_df["lag_2"] = ml_df["actual_net_revenue"].shift(2)
ml_df["lag_3"] = ml_df["actual_net_revenue"].shift(3)
ml_df["lag_12"] = ml_df["actual_net_revenue"].shift(12)

# Rolling features must use shifted values only
ml_df["rolling_mean_3"] = ml_df["actual_net_revenue"].shift(1).rolling(window=3).mean()
ml_df["rolling_mean_6"] = ml_df["actual_net_revenue"].shift(1).rolling(window=6).mean()
ml_df["rolling_std_3"] = ml_df["actual_net_revenue"].shift(1).rolling(window=3).std()

ml_df.head(15)

,date,Year,Quarter,Month,Month_Name,Year_Month,actual_net_revenue,time_idx,quarter_num,month_sin,month_cos,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3
0,2023-01-01,2023,Q1,1,January,2023-01,"408,301.70",0,1,0.50,0.87,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-02-01,2023,Q1,2,February,2023-02,"354,938.26",1,1,0.87,0.50,"408,301.70",NaN,NaN,NaN,NaN,NaN,NaN
2,2023-03-01,2023,Q1,3,March,2023-03,"371,116.33",2,1,1.00,0.00,"354,938.26","408,301.70",NaN,NaN,NaN,NaN,NaN
3,2023-04-01,2023,Q2,4,April,2023-04,"357,347.48",3,2,0.87,-0.50,"371,116.33","354,938.26","408,301.70",NaN,"378,118.76",NaN,"27,362.20"
4,2023-05-01,2023,Q2,5,May,2023-05,"360,818.12",4,2,0.50,-0.87,"357,347.48","371,116.33","354,938.26",NaN,"361,134.02",NaN,"8,728.45"
5,2023-06-01,2023,Q2,6,June,2023-06,"380,024.14",5,2,0.00,-1.00,"360,818.12","357,347.48","371,116.33",NaN,"363,093.98",NaN,"7,161.00"
6,2023-07-01,2023,Q3,7,July,2023-07,"380,199.86",6,3,-0.50,-0.87,"380,024.14","360,818.12","357,347.48",NaN,"366,063.25","372,091.01","12,214.39"
7,2023-08-01,2023,Q3,8,August,2023-08,"392,488.52",7,3,-0.87,-0.50,"380,199.86","380,024.14","360,818.12",NaN,"373,680.71","367,407.36","11,139.67"
8,2023-09-01,2023,Q3,9,September,2023-09,"371,716.66",8,3,-1.00,-0.00,"392,488.52","380,199.86","380,024.14",NaN,"384,237.51","373,665.74","7,146.13"
9,2023-10-01,2023,Q4,10,October,2023-10,"403,345.37",9,4,-0.87,0.50,"371,716.66","392,488.52","380,199.86",NaN,"381,468.35","373,765.80","10,443.87"


In [21]:
# Drop rows without lag features
feature_cols = [
    "time_idx",
    "Month",
    "quarter_num",
    "month_sin",
    "month_cos",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_12",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_std_3",
]

target_col = "actual_net_revenue"

ml_model_df = ml_df.dropna(subset=feature_cols + [target_col]).copy()

print("Original monthly rows:", len(ml_df))
print("ML-ready rows after lag creation:", len(ml_model_df))

ml_model_df[["date", "actual_net_revenue"] + feature_cols].head()

Original monthly rows: 36
ML-ready rows after lag creation: 24


,date,actual_net_revenue,time_idx,Month,quarter_num,month_sin,month_cos,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3
12,2024-01-01,"387,196.28",12,1,1,0.50,0.87,"429,787.17","383,222.99","403,345.37","408,301.70","405,451.84","393,460.09","23,353.45"
13,2024-02-01,"413,143.90",13,2,1,0.87,0.50,"387,196.28","429,787.17","383,222.99","354,938.26","400,068.81","394,626.17","25,813.41"
14,2024-03-01,"395,718.26",14,3,1,1.00,0.00,"413,143.90","387,196.28","429,787.17","371,116.33","410,042.45","398,068.73","21,464.16"
15,2024-04-01,"388,949.96",15,4,2,0.87,-0.50,"395,718.26","413,143.90","387,196.28","357,347.48","398,686.15","402,068.99","13,225.96"
16,2024-05-01,"422,960.68",16,5,2,0.50,-0.87,"388,949.96","395,718.26","413,143.90","360,818.12","399,270.71","399,669.76","12,482.05"


In [22]:
# Time-based ML train/test split
ml_train = ml_model_df[ml_model_df["date"].isin(train["date"])].copy()
ml_test = ml_model_df[ml_model_df["date"].isin(test["date"])].copy()

X_train_ml = ml_train[feature_cols]
y_train_ml = ml_train[target_col]

X_test_ml = ml_test[feature_cols]
y_test_ml = ml_test[target_col]

print("ML train rows:", len(ml_train))
print("ML test rows:", len(ml_test))
print("ML train period:", ml_train["date"].min().date(), "to", ml_train["date"].max().date())
print("ML test period:", ml_test["date"].min().date(), "to", ml_test["date"].max().date())

ML train rows: 18
ML test rows: 6
ML train period: 2024-01-01 to 2025-06-01
ML test period: 2025-07-01 to 2025-12-01


In [23]:
# Fit quick ML benchmark models

ml_models = {
    "Ridge Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=2,
        random_state=42,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=42,
    ),
}

ml_predictions = {}
ml_metrics = []

for model_name, model in ml_models.items():
    model.fit(X_train_ml, y_train_ml)
    pred = model.predict(X_test_ml)
    
    ml_predictions[model_name] = pred
    
    metrics = evaluate_forecast(
        y_test_ml,
        pred,
        model_name,
    )
    ml_metrics.append(metrics)

ml_forecast_metrics = pd.DataFrame(ml_metrics).sort_values("MAPE").reset_index(drop=True)

ml_forecast_metrics

,model,MAE,RMSE,MAPE
0,Random Forest,"39,687.74","44,252.82",0.10
1,Gradient Boosting,"44,309.81","47,764.08",0.11
2,Ridge Regression,"44,034.60","52,561.73",0.11


In [24]:
# Compare statistical models and ML benchmarks
combined_forecast_metrics = pd.concat(
    [
        forecast_metrics.assign(model_group="Time-Series / Baseline"),
        ml_forecast_metrics.assign(model_group="ML Benchmark"),
    ],
    ignore_index=True,
)

combined_forecast_metrics = combined_forecast_metrics.sort_values("MAPE").reset_index(drop=True)

combined_forecast_metrics

,model,MAE,RMSE,MAPE,model_group
0,Seasonal Naive,"34,422.12","43,539.32",0.08,Time-Series / Baseline
1,"SARIMA(0,1,1)(0,1,1,12)","38,148.42","47,558.61",0.09,Time-Series / Baseline
2,Holt-Winters,"36,417.62","41,767.31",0.09,Time-Series / Baseline
3,Random Forest,"39,687.74","44,252.82",0.10,ML Benchmark
4,Moving Average (3M),"42,522.63","49,002.65",0.10,Time-Series / Baseline
5,Gradient Boosting,"44,309.81","47,764.08",0.11,ML Benchmark
6,Ridge Regression,"44,034.60","52,561.73",0.11,ML Benchmark


In [25]:
best_ml_model_name = ml_forecast_metrics.iloc[0]["model"]
best_ml_mape = ml_forecast_metrics.iloc[0]["MAPE"]

print("Best ML benchmark:", best_ml_model_name)
print(f"Best ML benchmark MAPE: {best_ml_mape:.2%}")

ml_test_predictions = ml_test[["date"]].copy()
ml_test_predictions["ml_benchmark_forecast"] = ml_predictions[best_ml_model_name]

forecast_results = forecast_results.merge(
    ml_test_predictions,
    on="date",
    how="left",
)

forecast_results.head()

Best ML benchmark: Random Forest
Best ML benchmark MAPE: 9.91%


,date,Year,Quarter,Month,Month_Name,Year_Month,actual_net_revenue,seasonal_naive_forecast,moving_average_forecast,holt_winters_forecast,sarima_forecast,best_model,best_model_forecast,forecast_error,forecast_error_pct,ml_benchmark_forecast
0,2025-07-01,2025,Q3,7,July,2025-07,"373,404.97","399,964.40","424,551.23","419,409.19","405,435.60",Seasonal Naive,"399,964.40","-26,559.43",-0.07,"417,178.95"
1,2025-08-01,2025,Q3,8,August,2025-08,"454,648.33","372,987.42","422,792.51","421,596.21","364,973.67",Seasonal Naive,"372,987.42","81,660.91",0.18,"422,217.99"
2,2025-09-01,2025,Q3,9,September,2025-09,"413,041.17","411,851.11","412,127.46","414,986.00","424,524.13",Seasonal Naive,"411,851.11","1,190.06",0.00,"421,969.41"
3,2025-10-01,2025,Q4,10,October,2025-10,"388,158.81","399,160.10","419,823.73","434,779.05","394,877.90",Seasonal Naive,"399,160.10","-11,001.29",-0.03,"417,096.14"
4,2025-11-01,2025,Q4,11,November,2025-11,"352,684.08","404,944.21","418,247.90","419,932.31","410,729.53",Seasonal Naive,"404,944.21","-52,260.13",-0.15,"423,928.91"


In [26]:
final_model_ranking = combined_forecast_metrics.copy()
final_best_model_name = final_model_ranking.iloc[0]["model"]
final_best_model_group = final_model_ranking.iloc[0]["model_group"]
final_best_mape = final_model_ranking.iloc[0]["MAPE"]

print("Final best model:", final_best_model_name)
print("Model group:", final_best_model_group)
print(f"Final best MAPE: {final_best_mape:.2%}")

Final best model: Seasonal Naive
Model group: Time-Series / Baseline
Final best MAPE: 8.32%


# 13. Forecast Future

In [28]:
# Create future dates
FORECAST_HORIZON = 6

last_date = ts["date"].max()

future_dates = pd.date_range(
    start=last_date + pd.offsets.MonthBegin(1),
    periods=FORECAST_HORIZON,
    freq="MS"
)

future_df = pd.DataFrame({"date": future_dates})
future_df["Year"] = future_df["date"].dt.year
future_df["Quarter"] = "Q" + future_df["date"].dt.quarter.astype(str)
future_df["Month"] = future_df["date"].dt.month
future_df["Month_Name"] = future_df["date"].dt.month_name()
future_df["Year_Month"] = future_df["date"].dt.to_period("M").astype(str)

future_df

,date,Year,Quarter,Month,Month_Name,Year_Month
0,2026-01-01,2026,Q1,1,January,2026-01
1,2026-02-01,2026,Q1,2,February,2026-02
2,2026-03-01,2026,Q1,3,March,2026-03
3,2026-04-01,2026,Q2,4,April,2026-04
4,2026-05-01,2026,Q2,5,May,2026-05
5,2026-06-01,2026,Q2,6,June,2026-06


In [29]:
# Refit final best model and forecast future
full_series = ts.set_index("date")["actual_net_revenue"]

if final_best_model_name == "Seasonal Naive":
    seasonal_lookup_full = (
        ts[["Month", "actual_net_revenue"]]
        .groupby("Month", as_index=False)
        .tail(1)
        .set_index("Month")["actual_net_revenue"]
        .to_dict()
    )

    future_df["forecast_net_revenue"] = future_df["Month"].map(seasonal_lookup_full)
    future_df["forecast_net_revenue"] = future_df["forecast_net_revenue"].fillna(full_series.mean())

elif final_best_model_name == "Moving Average (3M)":
    future_df["forecast_net_revenue"] = recursive_moving_average_forecast(
        full_series.values,
        horizon=FORECAST_HORIZON,
        window=3,
    )

elif final_best_model_name == "Holt-Winters":
    final_hw_model = ExponentialSmoothing(
        full_series,
        trend="add",
        seasonal="add",
        seasonal_periods=12,
        initialization_method="estimated",
    )
    final_hw_fit = final_hw_model.fit(optimized=True)
    future_df["forecast_net_revenue"] = final_hw_fit.forecast(FORECAST_HORIZON).values

elif str(final_best_model_name).startswith("SARIMA") or str(final_best_model_name).startswith("ARIMA"):
    selected_candidate = None

    for candidate in sarima_candidates:
        if candidate["name"] == final_best_model_name:
            selected_candidate = candidate
            break

    if selected_candidate is None:
        raise ValueError(f"Selected SARIMA candidate not found: {final_best_model_name}")

    final_sarima_model = SARIMAX(
        full_series,
        order=selected_candidate["order"],
        seasonal_order=selected_candidate["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    final_sarima_fit = final_sarima_model.fit(disp=False)
    future_df["forecast_net_revenue"] = final_sarima_fit.forecast(steps=FORECAST_HORIZON).values

else:
    # Recursive ML future forecasting using leakage-safe lag features only.
    best_ml_model = ml_models[final_best_model_name]

    # Refit ML model on all available ML-ready data
    X_full_ml = ml_model_df[feature_cols]
    y_full_ml = ml_model_df[target_col]
    best_ml_model.fit(X_full_ml, y_full_ml)

    history = ts[["date", "actual_net_revenue"]].copy()
    future_forecasts = []

    for future_date in future_df["date"]:
        temp_history = history.copy().sort_values("date").reset_index(drop=True)

        month = future_date.month
        quarter_num = future_date.quarter
        time_idx = len(temp_history)

        feature_row = {
            "time_idx": time_idx,
            "Month": month,
            "quarter_num": quarter_num,
            "month_sin": np.sin(2 * np.pi * month / 12),
            "month_cos": np.cos(2 * np.pi * month / 12),
            "lag_1": temp_history["actual_net_revenue"].iloc[-1],
            "lag_2": temp_history["actual_net_revenue"].iloc[-2],
            "lag_3": temp_history["actual_net_revenue"].iloc[-3],
            "lag_12": temp_history["actual_net_revenue"].iloc[-12],
            "rolling_mean_3": temp_history["actual_net_revenue"].iloc[-3:].mean(),
            "rolling_mean_6": temp_history["actual_net_revenue"].iloc[-6:].mean(),
            "rolling_std_3": temp_history["actual_net_revenue"].iloc[-3:].std(),
        }

        X_future = pd.DataFrame([feature_row])[feature_cols]
        forecast_value = float(best_ml_model.predict(X_future)[0])
        future_forecasts.append(forecast_value)

        history = pd.concat(
            [
                history,
                pd.DataFrame({
                    "date": [future_date],
                    "actual_net_revenue": [forecast_value],
                })
            ],
            ignore_index=True,
        )

    future_df["forecast_net_revenue"] = future_forecasts

future_df

,date,Year,Quarter,Month,Month_Name,Year_Month,forecast_net_revenue
0,2026-01-01,2026,Q1,1,January,2026-01,"464,740.22"
1,2026-02-01,2026,Q1,2,February,2026-02,"381,580.46"
2,2026-03-01,2026,Q1,3,March,2026-03,"423,643.42"
3,2026-04-01,2026,Q2,4,April,2026-04,"429,827.38"
4,2026-05-01,2026,Q2,5,May,2026-05,"454,787.66"
5,2026-06-01,2026,Q2,6,June,2026-06,"389,038.64"


In [30]:
# Add planning bands using final best MAPE

forecast_error_rate = final_best_mape

future_df["lower_planning_band"] = future_df["forecast_net_revenue"] * (1 - forecast_error_rate)
future_df["upper_planning_band"] = future_df["forecast_net_revenue"] * (1 + forecast_error_rate)

future_df

,date,Year,Quarter,Month,Month_Name,Year_Month,forecast_net_revenue,lower_planning_band,upper_planning_band
0,2026-01-01,2026,Q1,1,January,2026-01,"464,740.22","426,078.12","503,402.32"
1,2026-02-01,2026,Q1,2,February,2026-02,"381,580.46","349,836.49","413,324.43"
2,2026-03-01,2026,Q1,3,March,2026-03,"423,643.42","388,400.20","458,886.64"
3,2026-04-01,2026,Q2,4,April,2026-04,"429,827.38","394,069.71","465,585.05"
4,2026-05-01,2026,Q2,5,May,2026-05,"454,787.66","416,953.52","492,621.80"
5,2026-06-01,2026,Q2,6,June,2026-06,"389,038.64","356,674.21","421,403.07"


In [31]:
# Forecast vs budget gap

REVENUE_TARGET_GROWTH = 0.08

historical_same_month = ts[["Month", "actual_net_revenue"]].copy()

# latest same-month actual available
historical_same_month = (
    historical_same_month
    .groupby("Month", as_index=False)
    .tail(1)
    .set_index("Month")["actual_net_revenue"]
    .to_dict()
)

future_df["budget_net_revenue"] = future_df["Month"].map(historical_same_month)
future_df["budget_net_revenue"] = future_df["budget_net_revenue"] * (1 + REVENUE_TARGET_GROWTH)

future_df["forecast_vs_budget_gap"] = (
    future_df["forecast_net_revenue"] - future_df["budget_net_revenue"]
)

future_df["forecast_vs_budget_gap_pct"] = (
    future_df["forecast_vs_budget_gap"] / future_df["budget_net_revenue"].replace(0, np.nan)
)

future_df["budget_gap_status"] = np.where(
    future_df["forecast_vs_budget_gap"] >= 0,
    "Above Budget Outlook",
    "Below Budget Outlook"
)

future_df

,date,Year,Quarter,Month,Month_Name,Year_Month,forecast_net_revenue,lower_planning_band,upper_planning_band,budget_net_revenue,forecast_vs_budget_gap,forecast_vs_budget_gap_pct,budget_gap_status
0,2026-01-01,2026,Q1,1,January,2026-01,"464,740.22","426,078.12","503,402.32","501,919.44","-37,179.22",-0.07,Below Budget Outlook
1,2026-02-01,2026,Q1,2,February,2026-02,"381,580.46","349,836.49","413,324.43","412,106.90","-30,526.44",-0.07,Below Budget Outlook
2,2026-03-01,2026,Q1,3,March,2026-03,"423,643.42","388,400.20","458,886.64","457,534.89","-33,891.47",-0.07,Below Budget Outlook
3,2026-04-01,2026,Q2,4,April,2026-04,"429,827.38","394,069.71","465,585.05","464,213.57","-34,386.19",-0.07,Below Budget Outlook
4,2026-05-01,2026,Q2,5,May,2026-05,"454,787.66","416,953.52","492,621.80","491,170.67","-36,383.01",-0.07,Below Budget Outlook
5,2026-06-01,2026,Q2,6,June,2026-06,"389,038.64","356,674.21","421,403.07","420,161.73","-31,123.09",-0.07,Below Budget Outlook


In [32]:
# Visualization of historical + future forecast

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=ts["date"],
        y=ts["actual_net_revenue"],
        mode="lines+markers",
        name="Historical Net Revenue",
    )
)

fig.add_trace(
    go.Scatter(
        x=future_df["date"],
        y=future_df["forecast_net_revenue"],
        mode="lines+markers",
        name=f"Future Forecast ({final_best_model_name})",
    )
)

fig.add_trace(
    go.Scatter(
        x=future_df["date"],
        y=future_df["upper_planning_band"],
        mode="lines",
        name="Upper Planning Band",
        line=dict(dash="dash"),
    )
)

fig.add_trace(
    go.Scatter(
        x=future_df["date"],
        y=future_df["lower_planning_band"],
        mode="lines",
        name="Lower Planning Band",
        line=dict(dash="dash"),
    )
)

fig.update_layout(
    title="Historical Net Revenue and 6-Month Forecast Outlook",
    xaxis_title="Month",
    yaxis_title="Net Revenue USD",
)

fig.show()

# 14. Model Comparison Chart (including ML model)

In [33]:
metrics_for_plot = combined_forecast_metrics.copy()
metrics_for_plot["MAPE_pct"] = metrics_for_plot["MAPE"] * 100

fig = px.bar(
    metrics_for_plot,
    x="model",
    y="MAPE_pct",
    color="model_group",
    text="MAPE_pct",
    title="Forecast Model Comparison Including ML Benchmark",
)

fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(
    xaxis_title="Model",
    yaxis_title="MAPE (%)",
)

fig.show()

# 15. Forecast Summary

In [34]:
next_month_forecast = future_df.iloc[0]["forecast_net_revenue"]
next_3_month_forecast = future_df.iloc[:3]["forecast_net_revenue"].sum()
next_6_month_forecast = future_df["forecast_net_revenue"].sum()

next_3_month_budget = future_df.iloc[:3]["budget_net_revenue"].sum()
next_6_month_budget = future_df["budget_net_revenue"].sum()

forecast_summary = pd.DataFrame({
    "metric": [
        "Best Forecast Model",
        "Best Model Group",
        "Test MAPE",
        "Next Month Forecast Revenue",
        "Next 3-Month Forecast Revenue",
        "Next 6-Month Forecast Revenue",
        "Next 3-Month Budget Gap",
        "Next 6-Month Budget Gap",
        "Forecast Horizon",
        "Planning Band Method",
    ],
    "value": [
        final_best_model_name,
        final_best_model_group,
        final_best_mape,
        next_month_forecast,
        next_3_month_forecast,
        next_6_month_forecast,
        next_3_month_forecast - next_3_month_budget,
        next_6_month_forecast - next_6_month_budget,
        f"{FORECAST_HORIZON} months",
        "Best model test MAPE applied as planning band",
    ]
})

forecast_summary

,metric,value
0,Best Forecast Model,Seasonal Naive
1,Best Model Group,Time-Series / Baseline
2,Test MAPE,0.08
3,Next Month Forecast Revenue,"464,740.22"
4,Next 3-Month Forecast Revenue,"1,269,964.10"
5,Next 6-Month Forecast Revenue,"2,543,617.78"
6,Next 3-Month Budget Gap,"-101,597.13"
7,Next 6-Month Budget Gap,"-203,489.42"
8,Forecast Horizon,6 months
9,Planning Band Method,Best model test MAPE applied as planning band


In [35]:
# Forecast interpretation
below_budget_months = int((future_df["budget_gap_status"] == "Below Budget Outlook").sum())
above_budget_months = int((future_df["budget_gap_status"] == "Above Budget Outlook").sum())

forecast_interpretation = pd.DataFrame({
    "insight_type": [
        "Best model",
        "Forecast accuracy",
        "Model selection",
        "ML benchmark result",
        "Short-term outlook",
        "Budget risk",
        "Planning caution",
    ],
    "insight": [
        f"{final_best_model_name} achieved the lowest test MAPE among the evaluated models.",
        f"The selected model produced a test MAPE of {final_best_mape:.2%}.",
        "The final model was selected based on out-of-sample performance rather than model complexity.",
        "The ML benchmark did not outperform the seasonal baseline, likely because the target is aggregated at monthly level with limited effective observations.",
        f"The next 6-month forecasted net revenue is {next_6_month_forecast:,.0f} USD.",
        f"{below_budget_months} out of {FORECAST_HORIZON} forecast months are below the simulated budget outlook.",
        "Forecast results should be reviewed together with promotion plans, pricing strategy, channel plans, and market conditions.",
    ]
})

forecast_interpretation

,insight_type,insight
0,Best model,Seasonal Naive achieved the lowest test MAPE a...
1,Forecast accuracy,The selected model produced a test MAPE of 8.32%.
2,Model selection,The final model was selected based on out-of-s...
3,ML benchmark result,The ML benchmark did not outperform the season...
4,Short-term outlook,"The next 6-month forecasted net revenue is 2,5..."
5,Budget risk,6 out of 6 forecast months are below the simul...
6,Planning caution,Forecast results should be reviewed together w...


In [37]:
ml_interpretation = pd.DataFrame({
    "insight_type": [
        "ML benchmark purpose",
        "Leakage control",
        "Data limitation",
        "Final selection rule",
    ],
    "insight": [
        "A quick ML benchmark was added to compare machine learning models against interpretable time-series baselines.",
        "Only calendar, lag, and rolling historical revenue features were used to avoid using current-month or future information.",
        "Because the monthly aggregate dataset is small, ML results should be interpreted as an experimental benchmark rather than a production forecasting model.",
        "The final model is selected based on test-period MAPE, not model complexity.",
    ],
})

ml_interpretation

,insight_type,insight
0,ML benchmark purpose,A quick ML benchmark was added to compare mach...
1,Leakage control,"Only calendar, lag, and rolling historical rev..."
2,Data limitation,Because the monthly aggregate dataset is small...
3,Final selection rule,The final model is selected based on test-peri...


# 16. Save Forecasting Artifacts

In [38]:
forecast_results.to_csv(PROCESSED_DATA_DIR / "forecast_results.csv", index=False)
combined_forecast_metrics.to_csv(PROCESSED_DATA_DIR / "forecast_metrics.csv", index=False)
future_df.to_csv(PROCESSED_DATA_DIR / "future_forecast.csv", index=False)
forecast_summary.to_csv(PROCESSED_DATA_DIR / "forecast_summary.csv", index=False)
forecast_interpretation.to_csv(PROCESSED_DATA_DIR / "forecast_interpretation.csv", index=False)
ml_interpretation.to_csv(PROCESSED_DATA_DIR / "ml_forecast_interpretation.csv", index=False)

print("Saved forecasting artifacts to:", PROCESSED_DATA_DIR)

Saved forecasting artifacts to: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new\data\processed


In [39]:
saved_files = [
    "forecast_results.csv",
    "forecast_metrics.csv",
    "future_forecast.csv",
    "forecast_summary.csv",
    "forecast_interpretation.csv",
    "ml_forecast_interpretation.csv",
]

for file in saved_files:
    path = PROCESSED_DATA_DIR / file
    temp = pd.read_csv(path)
    print(f"{file}: {temp.shape[0]:,} rows × {temp.shape[1]:,} columns")

forecast_results.csv: 6 rows × 16 columns
forecast_metrics.csv: 7 rows × 5 columns
future_forecast.csv: 6 rows × 13 columns
forecast_summary.csv: 10 rows × 2 columns
forecast_interpretation.csv: 7 rows × 2 columns
ml_forecast_interpretation.csv: 4 rows × 2 columns
